# Structural Similarity Index Measure (SSIM)

## Overview

The **Structural Similarity Index Measure (SSIM)** is an advanced metric used to measure the perceived quality of digital images and videos. Unlike traditional mathematical metrics, SSIM was specifically designed to mimic the **Human Visual System (HVS)**, focusing on structural degradation rather than simple pixel-by-pixel differences.

---

## Why Traditional Metrics Fall Short

Traditional metrics like **Mean Squared Error (MSE)** and **Peak Signal-to-Noise Ratio (PSNR)** operate on absolute mathematical differences between corresponding pixels. However, the human eye is notoriously insensitive to absolute pixel variations, but highly sensitive to changes in **structure, lighting, and contrast**.

For example, if an entire image is shifted slightly or uniformly brightened, MSE will register a massive error, even though a human viewer would perceive the image as nearly identical. SSIM resolves this by analyzing images in localized structural neighborhoods.

---

## The Three Core Components

SSIM evaluates two images ($x$ and $y$) by breaking the comparison down into three distinct perceptual pillars:

1. **Luminance ($l$):** Compares the average brightness of the images. Because human perception of brightness is non-linear, this component measures localized mean pixel intensities.
2. **Contrast ($c$):** Compares the standard deviation (variance) of the images. It evaluates how similar the contrast variations are between the reference and distorted versions.
3. **Structure ($s$):** Compares the local spatial patterns. It divides the image signals by their respective standard deviations to evaluate structural correlation independently of luminance and contrast.

Mathematically, these components are combined into the general formula:


$$\text{SSIM}(x,y) = [l(x,y)]^\alpha \cdot [c(x,y)]^\beta \cdot [s(x,y)]^\gamma$$


*(In standard implementations, the weights $\alpha$, $\beta$, and $\gamma$ are set to 1).*

---

## Score Interpretation

* **$1.0$ (or $100\%$):** The images are completely identical.
* **Lower values (approaching $0$):** Indicate significant structural distortion, loss of detail, or high degradation. Negative values can technically occur under extreme distortion, though rare in practice.

---

## Common Applications

* **Compression Evaluation:** Assessing quality loss in formats like JPEG, HEVC, or AV1.
* **Image Restoration:** Benchmarking algorithms designed for denoising, super-resolution, or inpainting.
* **Generative AI:** Measuring how closely synthetic images (from GANs or diffusion models) match original reference distributions.

In [4]:
import cv2
import numpy as np
from skimage.metrics import structural_similarity as ssim

# 1. Cargar una única imagen original
image = cv2.imread('original_image.png')

if image is None:
    print("Error: No se pudo cargar la imagen. Asegúrate de que la ruta sea correcta.")
    exit()

# Convertir la imagen a escala de grises
grayA = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

# 2. Generar una versión con ruido (Ruido Gaussiano)
mean = 0
sigma = 7  # Desviación estándar: un valor más alto genera más ruido
gauss = np.random.normal(mean, sigma, grayA.shape).astype(np.float32)

# Sumar el ruido a la imagen y limitar los valores entre 0 y 255
noisy_gray = np.clip(grayA.astype(np.float32) + gauss, 0, 255).astype(np.uint8)

# 3. Calcular el Structural Similarity Index (SSIM)
score, diff = ssim(grayA, noisy_gray, full=True)

# 4. Preparar la imagen de diferencia para visualizarla (0-255)
diff = (diff * 255).astype("uint8")

print(f"SSIM Score (Original vs. Con Ruido): {score:.4f}")

# 5. Mostrar las ventanas con los resultados
cv2.imshow("Original", grayA)
cv2.imshow("Con Ruido Sintetico", noisy_gray)
cv2.imshow("Mapa de Diferencia", diff)

cv2.waitKey(0)
cv2.destroyAllWindows()

SSIM Score (Original vs. Con Ruido): 0.8070


# What is a Difference Map?

**Definition:** A visual, pixel-by-pixel spatial representation showing **where and to what extent** two images differ from each other.

**Purpose:** While a global SSIM score gives a single overall numerical rating, a difference map pinpoints the exact locations of structural degradation or errors.

**Visual Interpretation:**
- **Bright / White areas:** High structural similarity (near-identical regions).
- **Dark / Black areas:** Low similarity, high distortion, or concentrated noise/artifacts.


**Main Use Cases:** Localizing compression artifacts, debugging image restoration algorithms, and evaluating generative AI or filtering outputs visually.